# De-noising diffusion experiments

In [1]:
from torch import Tensor, tensor, log
from torch.nn import Module


class SinPosEmbedding(Module):
    """Sinusoidal position embedding block.

    Parameters
    ----------
    dim : int
        No idea.
    theta : int
        The number of timesteps (I think).

    """

    def __init__(self, dim: int, theta: int = 10**4):
        """Build ``SinPosEmbedding``."""
        super().__init__()
        self._dim = dim
        self._theta = tensor(theta)

    def forward(self, batch: Tensor) -> Tensor:
        """Sinusoidally embed ``batch``."""
        half_dim = self._dim // 2
        embedding = log(self.theta) / (half_dim - 1)

In [24]:
from torch import linspace, cumprod
from torch.nn.functional import pad


class DiffusionModel(Module):
    """Wrapper class for the diffusion model.

    Parameters
    ----------
    time_steps : int, optional
        Number of time steps in the process.


    """

    def __init__(self, time_steps: int = 1000):
        """Build ``DiffusionModel``."""
        super().__init__()

        self.register_buffer("betas", linspace(1e-4, 1e-2, time_steps))
        self.register_buffer("alphas", 1.0 - self.betas)
        self.register_buffer("alphas_cp", cumprod(self.alphas, dim=0))
        self.register_buffer(
            "alphas_cp_prev",
            pad(self.alphas_cp[:-1], (1, 0), value=1.0),
        )
        self.register_buffer("sqrt_recip_alphas", (1 / self.alphas).sqrt())
        self.register_buffer("sqrt_alpha_cp", self.alphas_cp.sqrt())
        self.register_buffer("one_minus_sqrt_alphs_cp", (1 - self.alphas_cp).sqrt())
        self.register_buffer(
            "posterior_var",
            self.betas * (1.0 - self.alphas_cp_prev) / (1.0 - self.alphas_cp),
        )


DiffusionModel().sqrt_recip_alphas

tensor([1.0000, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001,
        1.0001, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001, 1.0001,
        1.0001, 1.0001, 1.0001, 1.0002, 1.0002, 1.0002, 1.0002, 1.0002, 1.0002,
        1.0002, 1.0002, 1.0002, 1.0002, 1.0002, 1.0002, 1.0002, 1.0002, 1.0002,
        1.0002, 1.0002, 1.0002, 1.0002, 1.0002, 1.0003, 1.0003, 1.0003, 1.0003,
        1.0003, 1.0003, 1.0003, 1.0003, 1.0003, 1.0003, 1.0003, 1.0003, 1.0003,
        1.0003, 1.0003, 1.0003, 1.0003, 1.0003, 1.0003, 1.0003, 1.0004, 1.0004,
        1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004,
        1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004, 1.0004,
        1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005,
        1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005, 1.0005,
        1.0005, 1.0005, 1.0006, 1.0006, 1.0006, 1.0006, 1.0006, 1.0006, 1.0006,
        1.0006, 1.0006, 1.0006, 1.0006, 